In [0]:
%pip install \
    azure-keyvault-secrets==4.7.0 \
    azure-identity==1.15.0 \
    azure-core==1.29.5 \
    azure-storage-file-datalake==12.14.0 \
    sseclient-py \
    openai \
    dotenv \
    confluent-kafka \
    great-expectations \
    altair==4.2.2 \
    redis

In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## PULSE — Wikipedia EventStreams 수신 + AI 규칙 생성
# MAGIC - Wikipedia SSE → Bronze 적재
# MAGIC - AI 규칙 생성 (최초 1회)
# MAGIC - GX 품질 검사

# COMMAND ----------
import os
import sys
import json
import requests
from sseclient import SSEClient
from datetime import datetime

# ── 환경 설정 ────────────────────────────────────────────
os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"
sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")

# ── vault 초기화 ─────────────────────────────────────────
import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager

vault = get_vault_manager()

# ── 연결 및 시크릿 로드 ──────────────────────────────────
storage_client    = vault.get_storage_client("datacopsadls")
kafka_producer    = vault.get_kafka_producer()

gx_openai_key        = vault.get_secret("gx-rulegen-openai-key")
gx_openai_endpoint   = vault.get_secret("gx-rulegen-openai-endpoint")
gx_openai_deployment = vault.get_secret("gx-rulegen-deployment-gpt-4-1-mini")
gx_openai_api_version = "2024-12-01-preview"

# ── 확인 ─────────────────────────────────────────────────
print("[OK] Key Vault 연결 완료")
print("[OK] ADLS 연결 완료")
print("[OK] Kafka Producer 연결 완료")
print("[OK] GX RuleGen OpenAI 설정 로드 완료")
print(f"[INFO] deployment = {gx_openai_deployment}")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, map_keys, col, udf
from pyspark.sql.types import MapType, StringType
import json as _json

spark = SparkSession.getActiveSession()
try:
    BRONZE_PATH = dbutils.widgets.get("bronze_path")
    print(f"[INFO] 전달받은 경로: {BRONZE_PATH}")
except:
    # 직접 실행 시 기본값 (테스트용)
    BRONZE_PATH = "abfss://bronze@datacopsadls.dfs.core.windows.net/wikipedia/"
    print(f"[INFO] 기본값 사용: {BRONZE_PATH}")

# source_name 추출 (경로 마지막 폴더명)
source_name = BRONZE_PATH.rstrip("/").split("/")[-1]
print(f"[INFO] 소스 이름: {source_name}")

df_init_spark = (
    spark.read
    .format("delta")
    .load(BRONZE_PATH)
    .limit(1000)
)

# UDF: JSON 문자열 → 모든 값을 String으로 flatten
@udf(MapType(StringType(), StringType()))
def flatten_json(raw):
    if raw is None:
        return None
    try:
        d = _json.loads(raw)
        result = {}
        for k, v in d.items():
            if isinstance(v, (dict, list)):
                result[k] = _json.dumps(v, ensure_ascii=False)
            elif v is None:
                result[k] = None
            else:
                result[k] = str(v)
        return result
    except Exception:
        return None

# raw_json → flatten Map으로 파싱
df_parsed = df_init_spark.withColumn("parsed", flatten_json(col("raw_json")))

# 전체 키 합집합 추출
all_keys = (
    df_parsed
    .select(map_keys("parsed").alias("keys"))
    .rdd.flatMap(lambda x: x["keys"])
    .distinct()
    .collect()
)

# 각 키를 컬럼으로 펼치기
for key in all_keys:
    df_parsed = df_parsed.withColumn(key, col("parsed")[key])

# 불필요한 컬럼 제거
cols_to_drop = ["parsed", "raw_json", "_source",
                "kafka_timestamp", "_bronze_loaded_at", "_kafka_topic"]
df_parsed = df_parsed.drop(*[c for c in cols_to_drop if c in df_parsed.columns])

# pandas 변환
df_init    = df_parsed.toPandas()
raw_events = df_init.to_dict("records")

print(f"[OK] Bronze에서 {len(raw_events)}건 로드 완료")
print(f"[INFO] 컬럼 ({len(df_init.columns)}개): {sorted(df_init.columns.tolist())}")

# null_rate 확인
null_rates = df_init.isnull().mean().sort_values()
print("\n[INFO] null_rate 상위 10개 (낮은 순):")
print(null_rates.head(10))

### 수집 데이터 gx 처리 

In [0]:
# COMMAND ----------
import pandas as pd

def detect_column_type(series: pd.Series) -> str:
    non_null = series.dropna()
    if non_null.empty:
        return "unknown"
    if pd.api.types.is_bool_dtype(non_null):
        return "boolean"
    if pd.api.types.is_numeric_dtype(non_null):
        return "numeric"
    sample = non_null.astype(str).head(20)
    parsed = pd.to_datetime(sample, errors="coerce", utc=True)
    if parsed.notna().mean() >= 0.8:
        return "timestamp"
    try:
        unique_ratio = non_null.nunique() / len(non_null)
    except TypeError:
        return "string"
    if unique_ratio < 0.05:
        return "categorical"
    return "string"

def safe_sample_values(series: pd.Series, n=3):
    values = series.dropna().head(n).tolist()
    result = []
    for v in values:
        try:
            json.dumps(v)
            result.append(v)
        except TypeError:
            result.append(str(v))
    return result

def safe_unique_count(series: pd.Series) -> int:
    try:
        return int(series.nunique())
    except TypeError:
        return int(series.astype(str).nunique())

def compute_null_correlations(df: pd.DataFrame, profile: dict) -> dict:
    """
    NULL이 많은 컬럼과 다른 컬럼 값의 상관관계 자동 계산
    도메인 무관하게 데이터 패턴에서 비즈니스 의미 추론
    예: length.old NULL → type=new와 98% 일치 → allow
    """
    nullable_cols = [
        col for col, info in profile.items()
        if 0.05 < info["null_rate"] < 0.95 and col in df.columns
    ]
    categorical_cols = [
    col for col, info in profile.items()
    if info["dtype"] in ["categorical", "boolean"]
    and info["null_rate"] < 0.05
    and col in df.columns
    and 1 < info["unique_count"] < 20  # 단일값/고카디널리티 제외
    ]

    for null_col in nullable_cols:
        null_mask = df[null_col].isna()
        if null_mask.sum() == 0:
            continue

        correlations = {}
        for cat_col in categorical_cols:
            if cat_col == null_col:
                continue
            when_null = df.loc[null_mask, cat_col].value_counts(normalize=True)
            for val, ratio in when_null.items():
                if ratio >= 0.8:  # 80% 이상 일치하면 의미있는 상관관계
                    correlations[f"{cat_col}={val}"] = round(float(ratio), 3)

        if correlations:
            profile[null_col]["null_when"] = correlations
            print(f"  [상관관계 발견] {null_col}: {correlations}")

    return profile

def auto_profile(data: list[dict]) -> dict:
    df = pd.json_normalize(data)
    profile = {}
    for col in df.columns:
        series = df[col]
        non_null = series.dropna()
        dtype = detect_column_type(series)
        col_info = {
            "dtype": dtype,
            "null_rate": round(series.isna().mean(), 3),
            "unique_count": safe_unique_count(non_null),
            "sample": safe_sample_values(series, n=3)
        }
        if dtype == "numeric" and not non_null.empty:
            col_info.update({
                "mean": round(float(non_null.mean()), 3),
                "std":  round(float(non_null.std()), 3),
            })
        profile[col] = col_info

    # 상관관계 계산 추가
    print("[INFO] NULL 상관관계 분석 중...")
    profile = compute_null_correlations(df, profile)

    return profile

profile = auto_profile(raw_events)
print(f"\n[OK] 컬럼 {len(profile)}개 분석 완료")
for col, info in list(profile.items())[:10]:
    print(f"  {col}: {info['dtype']} | null={info['null_rate']} | unique={info['unique_count']}")

### 칼럼 최적화 & 도메인 자동 감지 

In [0]:
# COMMAND ----------
# null_rate 0.95 이상 컬럼만 제외하고 전체 전달

slim_profile = {
    col: info
    for col, info in profile.items()
    if info["null_rate"] < 0.95
}

print(f"[INFO] 전체 {len(profile)}개 → AI 전달 {len(slim_profile)}개 컬럼")
print(f"[INFO] 제외된 컬럼 ({len(profile) - len(slim_profile)}개):")
for col, info in profile.items():
    if info["null_rate"] >= 0.95:
        print(f"  - {col}: null={info['null_rate']}")

print("\n[INFO] 비즈니스 의미 NULL 발견된 컬럼:")
for col, info in slim_profile.items():
    if "null_when" in info:
        print(f"  - {col}: {info['null_when']}")

# ── 도메인 자동 감지 ──────────────────────────────────────
from openai import AzureOpenAI

_client = AzureOpenAI(
    api_key=gx_openai_key,
    azure_endpoint=gx_openai_endpoint,
    api_version=gx_openai_api_version,
)

def detect_domain(profile: dict) -> dict:
    """
    프로파일 컬럼명 + 샘플값으로 도메인 자동 감지
    규칙 생성 전에 실행해서 도메인 컨텍스트 확보
    어떤 도메인 데이터든 자동 감지 가능
    """
    compact = json.dumps(
        {
            col: {
                "dtype": info["dtype"],
                "sample": info["sample"]
            }
            for col, info in list(profile.items())[:15]
        },
        ensure_ascii=False,
        indent=2
    )

    response = _client.chat.completions.create(
        model=gx_openai_deployment,
        messages=[
            {
                "role": "system",
                "content": """You are a data domain expert.
Identify the data domain from column names, dtypes, and sample values.
Return a JSON with:
- domain_name: short snake_case name (e.g. wikipedia_recentchange, nyc_taxi_trips, health_checkup)
- domain_description: one sentence describing the data
- key_columns: list of 3-5 most important columns
- data_characteristics: list of 2-3 notable characteristics

Return ONLY valid JSON. No markdown."""
            },
            {
                "role": "user",
                "content": f"Column profile:\n{compact}\n\nIdentify the domain."
            }
        ],
        temperature=0,
        max_tokens=200,
    )

    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {
            "domain_name": "unknown_domain",
            "domain_description": "Unknown domain",
            "key_columns": [],
            "data_characteristics": []
        }

# 실행
domain_info = detect_domain(slim_profile)
domain_name = domain_info["domain_name"]

print(f"\n[OK] 도메인 감지 완료")
print(f"  domain     : {domain_name}")
print(f"  description: {domain_info['domain_description']}")
print(f"  key_columns: {domain_info['key_columns']}")
print(f"  특성       : {domain_info['data_characteristics']}")

### AI 규칙 생성 프롬프트 작성

In [0]:
# COMMAND ----------

def build_stage1_prompt(profile: dict, domain_name: str) -> list[dict]:
    """
    1단계: 전체 컬럼 → NULL 전략 + 텍스트 품질 컬럼 식별
    null_when 필드로 도메인 무관하게 비즈니스 의미 NULL 자동 판단
    """
    compact_profile = json.dumps(profile, ensure_ascii=False, indent=2)

    system_prompt = """
You are a senior data quality engineer building a domain-agnostic automated data quality platform.
The platform works for ANY domain by inferring context from column names, dtypes, null rates, and sample values.

== NULL HANDLING STRATEGY ==
- drop             : critical identifier (name contains 'id', 'uuid', 'key', 'request_id') — drop row if null
- allow            : null has business meaning
- fill_default     : fill with fixed value (specify default_value)
- fill_mean        : fill with mean (normally distributed numeric)
- fill_median      : fill with median (skewed numeric)
- fill_mode        : fill with most frequent value (categorical/boolean)
- fill_forward     : fill with previous row value (time-ordered)
- fill_backward    : fill with next row value
- fill_interpolate : linear interpolation (ordered numeric)
- fill_conditional : fill based on another column value

== NULL CORRELATION RULE (HIGHEST PRIORITY) ==
If a column has "null_when" field:
- This means NULL is strongly correlated with a specific value in another column
- This NULL has BUSINESS MEANING → ALWAYS use "allow", never fill
- Example: null_when: {"type=new": 0.98} means NULL when type=new → new page creation → allow
- This rule overrides ALL other rules

== OTHER NULL RULES ==
- Identifier columns (name contains 'id', 'uuid', 'key', 'request_id'): ALWAYS "drop"
- Boolean columns (minor, patrolled, bot): "fill_mode"
- null_rate > 0.8 columns (log_*, subtypes): "allow"
- Free-text columns (comment, title, description, parsedcomment): "fill_default" with ""
- Timestamp columns: "fill_forward" or "drop", NEVER "fill_mean" or "fill_median"
- Categorical columns with null_rate > 0.1: "fill_mode"
- Skewed numeric (high std relative to mean): "fill_median"
- Normal numeric: "fill_mean"

== TEXT QUALITY RULES ==
For free-text columns (comment, description, title, parsedcomment, message, content, body):
- Identify which quality checks are needed based on column name and sample values
- checks: select from ["profanity", "spam", "hate_speech", "pii"]
- keywords.profanity: list 5-10 domain-relevant profanity/offensive words in the detected language
- keywords.hate_speech: list 5-10 domain-relevant hate speech terms in the detected language  
- patterns.spam: list regex patterns for spam detection (excessive URLs, repeated chars, etc.)
- patterns.pii: list regex patterns for domain-specific PII (email, phone, ID numbers, etc.)
- Only include checks that are relevant to the column's content based on sample values
- If no free-text columns exist, return empty list []

Return ONLY valid JSON. No markdown, no explanation, no comments.
"""

    user_prompt = f"""
Domain: {domain_name}

Column profile (dtype, null_rate, unique_count, sample, mean/std for numeric, null_when for correlated nulls):
{compact_profile}

Return:
{{
  "null_strategies": {{
    "<col>": {{
      "strategy": "drop|allow|fill_default|fill_mean|fill_median|fill_mode|fill_forward|fill_backward|fill_interpolate|fill_conditional",
      "default_value": null,
      "condition_column": null,
      "condition_map": null,
      "reason": "..."
    }}
  }},
  "text_quality_columns": [
    {{
      "column": "...",
      "checks": ["profanity", "spam", "hate_speech", "pii"],
      "keywords": {{
        "profanity": ["domain-specific profanity word1", "word2"],
        "hate_speech": ["domain-specific hate word1", "word2"]
      }},
      "patterns": {{
        "spam": ["regex_pattern1", "regex_pattern2"],
        "pii": ["regex_pattern_for_domain_specific_id"]
      }},
      "reason": "..."
    }}
  ]
}}
"""
    return [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_prompt.strip()}
    ]


def build_stage2_prompt(profile: dict, domain_name: str) -> list[dict]:
    """
    2단계: 핵심 컬럼만 → 검증 규칙 + 이상치 탐지
    null_rate < 0.1 또는 categorical/boolean/timestamp 위주
    """
    key_profile = {
        col: info for col, info in profile.items()
        if info["null_rate"] < 0.1
        or info["dtype"] in ["categorical", "boolean", "timestamp"]
    }
    compact_profile = json.dumps(key_profile, ensure_ascii=False, indent=2)

    system_prompt = """
You are a senior data quality engineer.
Generate stable validation rules and anomaly detection rules.

== VALIDATION RULES ==
- Do NOT generate min/max range rules (sample-dependent)
- Do NOT generate string length rules (sample-dependent)
- DO generate: not null checks, type checks, allowed value sets (cardinality < 10), datetime format
- severity: critical / warning / info
- One rule per expectation type per column, no duplicates
- Categorical with high unique_count (>10) or evolving over time: WARNING not CRITICAL
- For value_set rules on columns with unique_count > 3: use WARNING not CRITICAL
- If a column has "null_when" field in profile: do NOT generate expect_column_values_to_not_be_null for it

== ANOMALY RULES ==
- delta    : size/length change columns → flag extreme changes
- zscore   : normal numeric (use mean/std from profile) → flag beyond 3 std
- iqr      : skewed numeric → flag outside 1.5*IQR
- frequency: user/bot activity → flag abnormal rates
- Always specify numeric threshold where possible

Return ONLY valid JSON. No markdown, no explanation, no comments.
"""

    user_prompt = f"""
Domain: {domain_name}
Profile:
{compact_profile}

Return:
{{
  "suite_name": "{domain_name}_quality_suite",
  "domain": "{domain_name}",
  "generated_at": "2024-01-01T00:00:00Z",
  "expectations": [
    {{
      "rule_id": "RULE_{domain_name[:4].upper()}_001",
      "expectation_type": "...",
      "column": "...",
      "kwargs": {{}},
      "severity": "critical|warning|info",
      "error_code": "ERROR_COLUMNNAME_VIOLATION",
      "reason": "..."
    }}
  ],
  "anomaly_rules": [
    {{
      "rule_id": "RULE_{domain_name[:4].upper()}_010",
      "name": "...",
      "columns": ["..."],
      "method": "zscore|iqr|delta|frequency",
      "threshold": null,
      "severity": "critical|warning",
      "error_code": "ERROR_COLUMNNAME_ANOMALY",
      "reason": "..."
    }}
  ]
}}

rule_id 형식: RULE_{{도메인약자4자리}}_{{순번3자리}} (expectations 001~, anomaly_rules 010~)
error_code 형식: ERROR_{{컬럼명대문자}}_{{위반내용대문자}}
"""
    return [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_prompt.strip()}
    ]

# Redis 키 도메인 기반으로 자동 설정
CACHE_KEY_RULES  = f"gx_rules:{domain_name}"
CACHE_KEY_SCHEMA = f"gx_schema:{domain_name}"

# 프롬프트 생성
messages_stage1 = build_stage1_prompt(slim_profile, domain_name)
messages_stage2 = build_stage2_prompt(slim_profile, domain_name)
print(f"[OK] 1단계 프롬프트 준비 완료 (도메인: {domain_name})")
print(f"[OK] 2단계 프롬프트 준비 완료 (도메인: {domain_name})")
print(f"[INFO] 캐시 키: {CACHE_KEY_RULES}")

In [0]:
# COMMAND ----------
import redis
from redis.cluster import RedisCluster, ClusterNode
import hashlib
import json
from datetime import datetime

# ── Redis 연결 ────────────────────────────────────────────
redis_host = vault.get_secret("redis-host")
redis_password = vault.get_secret("redis-password")
redis_port = int(vault.get_secret("redis-port"))

# Azure Managed Redis endpoint 확인용
print(f"[INFO] Redis endpoint: {redis_host}:{redis_port}")

# Azure Managed Redis는 클러스터 모드 사용
# 주의:
# - Azure Managed Redis가 클러스터 노드 정보를 내부 IP로 반환할 수 있음
# - 이때 SSL 인증서는 도메인 기준인데, redis-py가 내부 IP로 검증하려고 해서
#   IP address mismatch 오류가 발생할 수 있음
r = RedisCluster(
    startup_nodes=[
        ClusterNode(redis_host, redis_port)
    ],
    password=redis_password,
    ssl=True,
    ssl_check_hostname=False,
    decode_responses=True,
    skip_full_coverage_check=True,
    socket_connect_timeout=10,
    socket_timeout=10,
)

r.ping()
print(f"[OK] Redis 연결 완료: {redis_host}:{redis_port}")


# ── 스키마 드리프트 감지 ──────────────────────────────────
def compute_schema_hash(profile: dict) -> str:
    schema_sig = {
        col: info["dtype"]
        for col, info in sorted(profile.items())
    }

    return hashlib.md5(
        json.dumps(schema_sig, sort_keys=True).encode()
    ).hexdigest()


def detect_schema_drift(profile: dict) -> dict:
    current_schema = {
        col: info["dtype"]
        for col, info in profile.items()
    }

    current_hash = compute_schema_hash(profile)

    stored_hash = r.get(CACHE_KEY_SCHEMA + ":hash")
    stored_schema = r.get(CACHE_KEY_SCHEMA + ":detail")

    # Redis에 기존 스키마 정보가 없으면 최초 실행으로 판단
    if not stored_hash or not stored_schema:
        return {
            "drifted": False,
            "is_first_run": True,
            "current_hash": current_hash,
            "changes": []
        }

    # 해시가 같으면 스키마 변경 없음
    if stored_hash == current_hash:
        return {
            "drifted": False,
            "is_first_run": False,
            "current_hash": current_hash,
            "changes": []
        }

    # 기존 스키마 상세 정보 로드
    prev_schema = json.loads(stored_schema)

    changes = []

    added = set(current_schema) - set(prev_schema)
    removed = set(prev_schema) - set(current_schema)

    for col in added:
        changes.append({
            "type": "added",
            "column": col,
            "dtype": current_schema[col]
        })

    for col in removed:
        changes.append({
            "type": "removed",
            "column": col,
            "dtype": prev_schema[col]
        })

    for col in set(current_schema) & set(prev_schema):
        if current_schema[col] != prev_schema[col]:
            changes.append({
                "type": "type_changed",
                "column": col,
                "from": prev_schema[col],
                "to": current_schema[col]
            })

    return {
        "drifted": True,
        "is_first_run": False,
        "current_hash": current_hash,
        "changes": changes
    }


# ── 실행 ─────────────────────────────────────────────────
drift_result = detect_schema_drift(slim_profile)

if drift_result["is_first_run"]:
    existing = r.get(CACHE_KEY_RULES)
    if existing:
        print("[INFO] 최초 스키마이지만 규칙 이미 존재 → 재사용")
        need_regenerate = False
    else:
        print("[INFO] 최초 실행 → 규칙 생성 필요")
        need_regenerate = True

elif drift_result["drifted"]:
    # type_changed만 있고 추가/삭제가 없으면 경고만 출력하고 규칙 재생성은 하지 않음
    real_changes = [
        c for c in drift_result["changes"]
        if c["type"] in ["added", "removed"]
    ]

    type_changes = [
        c for c in drift_result["changes"]
        if c["type"] == "type_changed"
    ]

    if type_changes and not real_changes:
        print(
            f"[INFO] 타입 변동 {len(type_changes)}개 감지 "
            "(샘플 크기 차이로 인한 변동 가능성, 규칙 유지)"
        )

        for c in type_changes:
            print(f"  ~ {c['column']}: {c['from']} → {c['to']}")

        need_regenerate = False

    else:
        print(
            f"[WARN] 스키마 드리프트 감지! "
            f"변경사항 {len(drift_result['changes'])}개:"
        )

        for c in drift_result["changes"]:
            if c["type"] == "added":
                print(f"  + 컬럼 추가: {c['column']} ({c['dtype']})")

            elif c["type"] == "removed":
                print(f"  - 컬럼 삭제: {c['column']}")

            elif c["type"] == "type_changed":
                print(
                    f"  ~ 타입 변경: {c['column']} "
                    f"({c['from']} → {c['to']})"
                )

        r.delete(CACHE_KEY_RULES)
        print("[OK] Redis 캐시 무효화 완료")

        need_regenerate = True

else:
    print("[INFO] 스키마 변경 없음 → 캐시 사용")
    need_regenerate = False

### AI 규칙 생성 

In [0]:
# COMMAND ----------
import json
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=gx_openai_key,
    azure_endpoint=gx_openai_endpoint,
    api_version=gx_openai_api_version,
)

INPUT_PRICE  = 0.40 / 1_000_000
OUTPUT_PRICE = 1.60 / 1_000_000

def call_ai(messages, label, max_tokens=8000):
    print(f"[INFO] {label} 생성 중...")
    response = client.chat.completions.create(
        model=gx_openai_deployment,
        messages=messages,
        temperature=0,
        max_tokens=max_tokens,
    )
    usage = response.usage
    cost  = (usage.prompt_tokens * INPUT_PRICE) + (usage.completion_tokens * OUTPUT_PRICE)
    print(f"  tokens: {usage.total_tokens} | cost: ${cost:.6f}")
    raw = response.choices[0].message.content.strip()

    token_info = {
        "prompt_tokens":     usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens":      usage.total_tokens,
        "cost_usd":          cost,
    }
    try:
        return json.loads(raw), token_info
    except json.JSONDecodeError as e:
        print(f"[WARN] JSON 파싱 실패: {e}")
        print(f"[RAW] {raw[:300]}")
        return {}, token_info

# ── 캐시 히트 확인 ───────────────────────────────────────
cached = r.get(CACHE_KEY_RULES)

if cached and not need_regenerate:
    gx_rules = json.loads(cached)
    print("[OK] Redis 캐시에서 규칙 로드 (AI 호출 없음)")
    print(f"  NULL 전략  : {len(gx_rules.get('null_strategies', {}))}개")
    print(f"  검증 규칙  : {len(gx_rules.get('expectations', []))}개")
    print(f"  이상치 탐지: {len(gx_rules.get('anomaly_rules', []))}개")
    print(f"  텍스트 품질: {len(gx_rules.get('text_quality_columns', []))}개")

else:
    # ── AI 호출 ───────────────────────────────────────────
    stage1, token1 = call_ai(messages_stage1, "1단계 (NULL 전략 + 텍스트 품질)")
    stage2, token2 = call_ai(messages_stage2, "2단계 (검증 규칙 + 이상치 탐지)")

    gx_rules = {
        "suite_name"          : stage2.get("suite_name", ""),
        "domain"              : stage2.get("domain", ""),
        "version"             : "1.0",
        "generated_at"        : datetime.utcnow().isoformat() + "Z",
        "null_strategies"     : stage1.get("null_strategies", {}),
        "expectations"        : stage2.get("expectations", []),
        "anomaly_rules"       : stage2.get("anomaly_rules", []),
        "text_quality_columns": stage1.get("text_quality_columns", []),
    }

    total_cost   = token1["cost_usd"] + token2["cost_usd"]
    total_tokens = token1["total_tokens"] + token2["total_tokens"]
    print(f"\n[OK] 전체 규칙 생성 완료 | 총 토큰: {total_tokens} | 총 비용: ${total_cost:.6f}")
    print(f"  NULL 전략  : {len(gx_rules['null_strategies'])}개")
    print(f"  검증 규칙  : {len(gx_rules['expectations'])}개")
    print(f"  이상치 탐지: {len(gx_rules['anomaly_rules'])}개")
    print(f"  텍스트 품질: {len(gx_rules['text_quality_columns'])}개")

    # ── Redis에 저장 (30일 TTL) ───────────────────────────
    TTL = 60 * 60 * 24 * 30
    current_schema = {col: info["dtype"] for col, info in slim_profile.items()}
    gx_rules["slim_profile"] = slim_profile

    existing = r.get(CACHE_KEY_RULES)
    if existing:
        old_rules    = json.loads(existing)
        old_version  = old_rules.get("version", "1.0")
        major, minor = old_version.split(".")
        gx_rules["version"] = f"{major}.{int(minor) + 1}"
    else:
        gx_rules["version"] = "1.0"

    r.set(CACHE_KEY_RULES,              json.dumps(gx_rules, ensure_ascii=False), ex=TTL)
    r.set(CACHE_KEY_SCHEMA + ":hash",   drift_result["current_hash"],             ex=TTL)
    r.set(CACHE_KEY_SCHEMA + ":detail", json.dumps(current_schema),               ex=TTL)
    print("[OK] Redis에 규칙 + 스키마 저장 완료 (TTL: 30일)")
    print(f"  캐시 키: {CACHE_KEY_RULES}")

    # ── AI 비용 로그 저장 ─────────────────────────────────
    generated_at = datetime.utcnow().isoformat() + "Z"
    today        = datetime.utcnow().strftime("%Y-%m-%d")

    ai_cost_log = {
        "window_start":         generated_at,
        "domain_name":          domain_name,
        "rule_version":         gx_rules.get("version", "1.0"),
        "generated_rule_count": len(gx_rules.get("expectations", [])) + len(gx_rules.get("anomaly_rules", [])),
        "total_tokens":         total_tokens,
        "estimated_cost_usd":   round(total_cost, 6),
        "stage1": {
            "prompt_tokens":     token1["prompt_tokens"],
            "completion_tokens": token1["completion_tokens"],
            "total_tokens":      token1["total_tokens"],
            "cost_usd":          round(token1["cost_usd"], 6),
        },
        "stage2": {
            "prompt_tokens":     token2["prompt_tokens"],
            "completion_tokens": token2["completion_tokens"],
            "total_tokens":      token2["total_tokens"],
            "cost_usd":          round(token2["cost_usd"], 6),
        },
    }

    log_path = (
        f"abfss://logs@datacopsadls.dfs.core.windows.net"
        f"/ai_cost_logs/date={today}/domain={domain_name}"
        f"/run_{generated_at.replace(':', '-')}.json"
    )
    log_json = json.dumps(ai_cost_log, ensure_ascii=False, indent=2)
    spark.createDataFrame([(log_json,)], ["log"]) \
         .write.mode("overwrite").text(log_path)
    print(f"  [OK] AI 비용 로그 저장: {log_path}")

# ── 01 완료 ───────────────────────────────────────────────
print("\n[완료] 01_rules_generator 실행 완료")
print(f"  도메인    : {domain_name}")
print(f"  캐시 키   : {CACHE_KEY_RULES}")
print(f"  규칙      : {len(gx_rules.get('expectations', []))}개")
print(f"  이상치    : {len(gx_rules.get('anomaly_rules', []))}개")
print("\n→ 02_stream_bronze2silver 실행 가능")